# Side-by-Side Search Pairs

`SideBySideSearchPair` turns a batch of single `224x224` images into
**`target | distractor`** side-by-side search displays `(B, 3, 224, 448)` for a
minimal visual-search task.

For each sample `i`:
- its **own** image is the **target**, placed on side `target_side[i]`
  (`0=left`, `1=right`),
- a within-batch **partner**'s image is the **different-class distractor** on
  the other side.

Outputs written to the batch:
- `image` -> `(B, 3, 224, 448)` composite
- `label` -> the **target** label (sample `i`'s own label, unchanged)
- `target_side` -> `0=left` / `1=right`
- `distractor_label` -> partner's class label
- `valid` -> usable-trial mask

Pipeline placement: an `after_batch_transform` on **[0,1] RGB** (NOT normalized
- the consumer normalizes per-checkpoint downstream).

This notebook visualizes the pairs (green = target, red = distractor) with the
labels, in both pairing modes.

In [ ]:
LITDATA_VAL_PATH = "s3://visionlab-datasets/imagenet1k/pre-processed/s256-l512-jpgbytes-q100-streaming/val/"

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from slipstream import SlipstreamDataset, SlipstreamLoader, DecodeRandomResizedCrop
from slipstream.transforms import SideBySideSearchPair, ToFloatDiv

dataset = SlipstreamDataset(remote_dir=LITDATA_VAL_PATH, decode_images=False)
print(f"Dataset: {len(dataset):,} samples")

In [ ]:
def load_batch(dataset, size=224, batch_size=8):
    """Load one batch as float-in-[0,1] RGB images plus int64 labels."""
    dec = DecodeRandomResizedCrop(size=size, to_tensor=True, permute=True)
    loader = SlipstreamLoader(
        dataset, batch_size=batch_size, shuffle=True, seed=0,
        pipelines={'image': [dec]}, verbose=False,
    )
    batch = next(iter(loader))
    loader.shutdown()
    return {
        'image': batch['image'].float() / 255.0,
        'label': batch['label'].long(),
    }


def show_pairs(out, transform, suptitle=None, max_cols=8):
    """Show side-by-side composites; outline target (green) / distractor (red).

    Title per panel: target label (= `label`), distractor label, side, valid.
    """
    comp = out['image']
    B = comp.shape[0]
    W = comp.shape[-1] // 2
    label = out['label'].cpu().numpy()
    side = transform.last_target_side          # 0=left, 1=right
    dlabel = transform.last_distractor_label
    valid = transform.last_valid

    n = min(max_cols, B)
    fig, axes = plt.subplots(1, n, figsize=(3.0 * n, 3.4))
    axes = np.atleast_1d(axes)
    for i in range(n):
        img = comp[i].detach().float().permute(1, 2, 0).clamp(0, 1).cpu().numpy()
        ax = axes[i]
        ax.imshow(img)
        ax.axvline(W - 0.5, color='white', lw=1.5)   # the A|B seam
        # green box on the target half, red on the distractor half
        tgt_x0 = 0 if side[i] == 0 else W
        dis_x0 = W if side[i] == 0 else 0
        ax.add_patch(Rectangle((tgt_x0, 0), W - 1, img.shape[0] - 1,
                               fill=False, edgecolor='lime', lw=3))
        ax.add_patch(Rectangle((dis_x0, 0), W - 1, img.shape[0] - 1,
                               fill=False, edgecolor='red', lw=3))
        side_str = 'LEFT' if side[i] == 0 else 'RIGHT'
        v = 'valid' if valid[i] else 'INVALID'
        ax.set_title(
            f"target c={label[i]} ({side_str})\n"
            f"distractor c={int(dlabel[i])}  [{v}]",
            fontsize=9,
        )
        ax.axis('off')
    if suptitle:
        fig.suptitle(suptitle, fontsize=13, fontweight='bold', y=1.04)
    # legend
    handles = [Rectangle((0, 0), 1, 1, fill=False, edgecolor='lime', lw=3),
               Rectangle((0, 0), 1, 1, fill=False, edgecolor='red', lw=3)]
    fig.legend(handles, ['target (goal)', 'distractor'],
               loc='lower center', ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.06))
    plt.tight_layout()
    plt.show()


batch = load_batch(dataset, size=224, batch_size=8)
print(f"image: {tuple(batch['image'].shape)}, label: {tuple(batch['label'].shape)}")

## Default mode — random derangement (instance search)

Partner = a seeded random within-batch permutation with **no fixed points**, so
no sample is paired with itself and there is no fixed class->class mapping. On
ImageNet-scale batches ~all pairs are different-class; the rare same-class pair
is just a harder instance-discrimination trial. `valid` is all-`True`.

Green box = the **target** (the sample's own image, the goal). Red box = the
**distractor** (a different image from the batch).

In [ ]:
t = SideBySideSearchPair(num_classes=1000, seed=2026)
out = t({'image': batch['image'].clone(), 'label': batch['label'].clone()})

print("composite:", tuple(out['image'].shape))
print("target_side  :", out['target_side'].tolist())
print("label (tgt)  :", out['label'].tolist())
print("distractor   :", out['distractor_label'].tolist())
print("valid        :", out['valid'].tolist())
print("partner index:", t.last_partner_index.tolist())

show_pairs(out, t, suptitle='SideBySideSearchPair (default: random derangement)')
print(repr(t))

In [ ]:
from PIL import Image

img = out['image'][0].detach().float().permute(1, 2, 0).clamp(0, 1).cpu().mul(255).numpy().astype(np.uint8)
Image.fromarray(img)

## Sanity check — halves match their source crops

The target half must equal the sample's own crop, and the distractor half must
equal the partner's crop. We verify against the original (pre-composite) batch.

In [ ]:
src = batch['image']                       # original single crops
b2 = {'image': batch['image'].clone(), 'label': batch['label'].clone()}
t2 = SideBySideSearchPair(num_classes=1000, seed=2026)
out2 = t2(b2)

W = src.shape[-1]
comp = out2['image']
left, right = comp[:, :, :, :W], comp[:, :, :, W:]
side = t2.last_target_side
partner = t2.last_partner_index

ok = True
for i in range(src.shape[0]):
    tgt_half, dis_half = (left[i], right[i]) if side[i] == 0 else (right[i], left[i])
    ok &= torch.equal(tgt_half, src[i])
    ok &= torch.equal(dis_half, src[partner[i]])
print("target half == own crop AND distractor half == partner crop for all samples:", bool(ok))

## `require_different_class=True` — category-prototype search

When the search template is a **category prototype**, a same-class distractor is
ill-posed, so we require a different-class partner. Pairing chunks same-class
samples adjacent (ordered by each label's *first appearance* in the shuffled
batch -> random per batch, no fixed class->class mapping), then pairs a
half-batch apart. `valid[i]` is `True` iff the partner is a different class
(only false in the pathological case of one class dominating the batch).

Here we force same-class collisions with a tiny label set so you can see an
`INVALID` panel get flagged.

In [ ]:
# few classes + larger batch => some same-class collisions => some INVALID trials
bc = load_batch(dataset, size=224, batch_size=8)
K = 3
bc['label'] = torch.randint(0, K, (bc['label'].shape[0],))

t3 = SideBySideSearchPair(num_classes=K, seed=7, require_different_class=True)
out3 = t3({'image': bc['image'].clone(), 'label': bc['label'].clone()})

print("labels       :", bc['label'].tolist())
print("partner index:", t3.last_partner_index.tolist())
print("distractor   :", out3['distractor_label'].tolist())
print("valid        :", out3['valid'].tolist())
print(f"valid fraction: {out3['valid'].float().mean().item():.2f}")

show_pairs(out3, t3,
           suptitle='require_different_class=True — distractor != target class (else INVALID)')

## Target-side distribution follows `p_left`

`p_left` controls how often the target lands on the left. Over many batches the
empirical left-fraction should track it.

In [ ]:
for p in (0.5, 0.8):
    t = SideBySideSearchPair(num_classes=1000, seed=0, p_left=p)
    sides = []
    for s in range(200):
        g = torch.Generator().manual_seed(s)
        fake = {'image': torch.rand(64, 3, 8, 8, generator=g),
                'label': torch.randint(0, 1000, (64,), generator=g)}
        sides.append(t(fake)['target_side'].cpu().numpy())
    frac_left = (np.concatenate(sides) == 0).mean()
    print(f"p_left={p}: empirical left fraction = {frac_left:.3f}")